<a href="https://colab.research.google.com/github/comparativechrono/computational_biology_notebooks/blob/main/Bioinformatics/Algorithms_8_consensus_and_conservation_in_Multiple_Sequence_Alignments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Calculating consensus and conservation scores from a Multiple Sequence Alignment

Methods for assembling multiple sequence alignments have been described in the lecture. Here the application of conservation measures to the resulting MSA is considered.

Similarity measures need a substitution table which can be a matrix or as here as a Python dictionary of dictionaries.

Gap penalties will also need to be considered for the MSA. And for some sequence comparisons n an MSA there can be a gap aligned with a gap.

## BLOSUM62

Here is the BLOSUM62 matrix, stored as a dictionary of dictionaries:

In [ ]:
# run this cell to store the BLOSUM62 matrix (as a constant)
BLOSUM62 = {'A':{'A': 4,'R':-1,'N':-2,'D':-2,'C': 0,'Q':-1,'E':-1,'G': 0,'H':-2,'I':-1,
                 'L':-1,'K':-1,'M':-1,'F':-2,'P':-1,'S': 1,'T': 0,'W':-3,'Y':-2,'V': 0},
            'R':{'A':-1,'R': 5,'N': 0,'D':-2,'C':-3,'Q': 1,'E': 0,'G':-2,'H': 0,'I':-3,
                 'L':-2,'K': 2,'M':-1,'F':-3,'P':-2,'S':-1,'T':-1,'W':-3,'Y':-2,'V':-3},
            'N':{'A':-2,'R': 0,'N': 6,'D': 1,'C':-3,'Q': 0,'E': 0,'G': 0,'H': 1,'I':-3,
                 'L':-3,'K': 0,'M':-2,'F':-3,'P':-2,'S': 1,'T': 0,'W':-4,'Y':-2,'V':-3},
            'D':{'A':-2,'R':-2,'N': 1,'D': 6,'C':-3,'Q': 0,'E': 2,'G':-1,'H':-1,'I':-3,
                 'L':-4,'K':-1,'M':-3,'F':-3,'P':-1,'S': 0,'T':-1,'W':-4,'Y':-3,'V':-3},
            'C':{'A': 0,'R':-3,'N':-3,'D':-3,'C': 9,'Q':-3,'E':-4,'G':-3,'H':-3,'I':-1,
                 'L':-1,'K':-3,'M':-1,'F':-2,'P':-3,'S':-1,'T':-1,'W':-2,'Y':-2,'V':-1},
            'Q':{'A':-1,'R': 1,'N': 0,'D': 0,'C':-3,'Q': 5,'E': 2,'G':-2,'H': 0,'I':-3,
                 'L':-2,'K': 1,'M': 0,'F':-3,'P':-1,'S': 0,'T':-1,'W':-2,'Y':-1,'V':-2},
            'E':{'A':-1,'R': 0,'N': 0,'D': 2,'C':-4,'Q': 2,'E': 5,'G':-2,'H': 0,'I':-3,
                 'L':-3,'K': 1,'M':-2,'F':-3,'P':-1,'S': 0,'T':-1,'W':-3,'Y':-2,'V':-2},
            'G':{'A': 0,'R':-2,'N': 0,'D':-1,'C':-3,'Q':-2,'E':-2,'G': 6,'H':-2,'I':-4,
                 'L':-4,'K':-2,'M':-3,'F':-3,'P':-2,'S': 0,'T':-2,'W':-2,'Y':-3,'V':-3},
            'H':{'A':-2,'R': 0,'N': 1,'D':-1,'C':-3,'Q': 0,'E': 0,'G':-2,'H': 8,'I':-3,
                 'L':-3,'K':-1,'M':-2,'F':-1,'P':-2,'S':-1,'T':-2,'W':-2,'Y': 2,'V':-3},
            'I':{'A':-1,'R':-3,'N':-3,'D':-3,'C':-1,'Q':-3,'E':-3,'G':-4,'H':-3,'I': 4,
                 'L': 2,'K':-3,'M': 1,'F': 0,'P':-3,'S':-2,'T':-1,'W':-3,'Y':-1,'V': 3},
            'L':{'A':-1,'R':-2,'N':-3,'D':-4,'C':-1,'Q':-2,'E':-3,'G':-4,'H':-3,'I': 2,
                 'L': 4,'K':-2,'M': 2,'F': 0,'P':-3,'S':-2,'T':-1,'W':-2,'Y':-1,'V': 1},
            'K':{'A':-1,'R': 2,'N': 0,'D':-1,'C':-3,'Q': 1,'E': 1,'G':-2,'H':-1,'I':-3,
                 'L':-2,'K': 5,'M':-1,'F':-3,'P':-1,'S': 0,'T':-1,'W':-3,'Y':-2,'V':-2},
            'M':{'A':-1,'R':-1,'N':-2,'D':-3,'C':-1,'Q': 0,'E':-2,'G':-3,'H':-2,'I': 1,
                 'L': 2,'K':-1,'M': 5,'F': 0,'P':-2,'S':-1,'T':-1,'W':-1,'Y':-1,'V': 1},
            'F':{'A':-2,'R':-3,'N':-3,'D':-3,'C':-2,'Q':-3,'E':-3,'G':-3,'H':-1,'I': 0,
                 'L': 0,'K':-3,'M': 0,'F': 6,'P':-4,'S':-2,'T':-2,'W': 1,'Y': 3,'V':-1},
            'P':{'A':-1,'R':-2,'N':-2,'D':-1,'C':-3,'Q':-1,'E':-1,'G':-2,'H':-2,'I':-3,
                 'L':-3,'K':-1,'M':-2,'F':-4,'P': 7,'S':-1,'T':-1,'W':-4,'Y':-3,'V':-2},
            'S':{'A': 1,'R':-1,'N': 1,'D': 0,'C':-1,'Q': 0,'E': 0,'G': 0,'H':-1,'I':-2,
                 'L':-2,'K': 0,'M':-1,'F':-2,'P':-1,'S': 4,'T': 1,'W':-3,'Y':-2,'V':-2},
            'T':{'A': 0,'R':-1,'N': 0,'D':-1,'C':-1,'Q':-1,'E':-1,'G':-2,'H':-2,'I':-1,
                 'L':-1,'K':-1,'M':-1,'F':-2,'P':-1,'S': 1,'T': 5,'W':-2,'Y':-2,'V': 0},
            'W':{'A':-3,'R':-3,'N':-4,'D':-4,'C':-2,'Q':-2,'E':-3,'G':-2,'H':-2,'I':-3,
                 'L':-2,'K':-3,'M':-1,'F': 1,'P':-4,'S':-3,'T':-2,'W':11,'Y': 2,'V':-3},
            'Y':{'A':-2,'R':-2,'N':-2,'D':-3,'C':-2,'Q':-1,'E':-2,'G':-3,'H': 2,'I':-1,
                 'L':-1,'K':-2,'M':-1,'F': 3,'P':-3,'S':-2,'T':-2,'W': 2,'Y': 7,'V':-1},
            'V':{'A': 0,'R':-3,'N':-3,'D':-3,'C':-1,'Q':-2,'E':-2,'G':-3,'H':-3,'I': 3,
                 'L': 1,'K':-2,'M': 1,'F':-1,'P':-2,'S':-2,'T': 0,'W':-3,'Y':-1,'V': 4}
           }

In order to look up a value from the matrix we can write Python like:

In [ ]:
# run this cell to print out the BLOSUM score for substituting 'W' with 'Y'
print(BLOSUM62['W']['Y'])

In [ ]:
# now it is your turn what is the BLOSUM62 for substituting 'Y' with 'W'?
# write python to look up the value

Is the value what you would expect? Why?

In Software development testing is always a good idea. The BLOSUM62 matrix is available from:
https://bio.libretexts.org/Bookshelves/Cell_and_Molecular_Biology/Book%3A_Investigations_in_Molecular_Cell_Biology_(O'Connor)/9%3A_Protein_Conservation/9.4%3A_BLOSUM62_scoring_matrix_for_amino_acid_substitutions

Using information from the table check 5 different values in BLOSUM62 using assert statements to check 6 values BLOSUM62 matrix.

In [ ]:
# complete the following python to check 6 values from BLOSUM62
assert BLOSUM62['W']['Y'] == 2
assert BLOSUM62['Y']['W'] == 2
# add 4 more assert statements

The BLOSUM62 matrix is symmetric and should have values for each of the coded amino acids (one letter code `ACDEFGHIKLMNPQRSTVWY`). Write Python to check that `BLOSUM62` constant is symmetic using `assert` check.

In [ ]:
# your python checking that BLOSUM62 is symmetic
coded_aas = 'ACDEFGHIKLMNPQRSTVWY'

## Using BLOSUM62 to score and alignment

For MSAs a pairwise similarity measure can be calculated by summing the scores at each position from the substitution table. Remember that the summing is actually equivalent to multiplying the chances of possible changes (or conservation). But that the odds for these have been made into logs which can be added.

Here is a function to use the dictionary values.

In [ ]:
# run this cell to define function calcPairwiseSimilarity
def calcPairwiseSimilarity(seqA, seqB, substTable):
    """
    calculate total score for aligning seqA and seqB using substTable
    (no gaps allowed)
    """
    numPositions = min(len(seqA), len(seqB))

    totalScore = 0

    for position in range(numPositions):
        resA = seqA[position]
        resB = seqB[position]
        totalScore += substTable[resA][resB]

    return totalScore


In [ ]:
# run this cell to check that calcPairwiseSimilarity works as expected.
assert calcPairwiseSimilarity('W', 'W', BLOSUM62) == BLOSUM62['W']['W']
assert calcPairwiseSimilarity('WWW', 'WWW', BLOSUM62) == 3*BLOSUM62['W']['W']
assert calcPairwiseSimilarity('A', 'N', BLOSUM62) == BLOSUM62['A']['N']
assert calcPairwiseSimilarity('AA', 'NN', BLOSUM62) == 2*BLOSUM62['A']['N']
assert calcPairwiseSimilarity('WAA', 'WNN', BLOSUM62) == BLOSUM62['W']['W'] + 2*BLOSUM62['A']['N']

In [ ]:
firstSeq = 'TTCCPSIVARSNFNVCRLPGTPEALCATYTGCIIIPGATCPGDYAN'
secondSeq = 'KSCCPTTAARNQYNICRLPGTPRPVCAALSGCKIISGTGCPPGYRH'

In [ ]:
pairwise = calcPairwiseSimilarity(firstSeq, secondSeq, BLOSUM62)
print(pairwise, '(should be 146!)')

### Advanced optional exercise
* `calcPairwiseSimilarity` is code that is not PEP8 compliant
* It also uses `C` like syntax to access string elements by index

Rewrite it to be PEP8 compliant and to use Python syntax.

## Scoring a pairwise Sequence alignment
To get the overall score for a multiple sequence alignment, every pair would be taken in turn from the alignment.

Pairs from multiple alignments can have gaps in either sequences in the pair. But also possibly gaps in both sequences.

So modify the function to give a score of -1 for a single gap '-'. And a score of 0 for a gap in each sequence aligned with each other. Since two such gaps come from matching other pairs in the multiple sequence alignments they are given a score of 0.

In [ ]:
### your code for calcPairwiseSimilarityGapped

In [ ]:
# use this cell to check your calcPairwiseSimilarityGapped function.
assert calcPairwiseSimilarityGapped('-', '-', BLOSUM62) == 0
assert calcPairwiseSimilarityGapped('-', 'A', BLOSUM62) == -1
assert calcPairwiseSimilarityGapped('W', 'W', BLOSUM62) == 11
assert calcPairwiseSimilarityGapped('WWW', 'WWW', BLOSUM62) == 33
assert calcPairwiseSimilarityGapped('WAA', 'WNN', BLOSUM62) == 7
assert calcPairwiseSimilarityGapped('WWW', 'W-W', BLOSUM62) == 22-1

### Sequence profile
Several uses for a sequence profile have been mentioned: searching a database, or standing in for a group of sequences in a multiple sequence alignment fro example.

As an example for a profile calculation, here is a small multiple sequence alignment. It is from a BLAST search for sequences similar to the firstSeq above. This was taken from a small robust protein from plant seeds called CRAMBIN (https://www.wikiwand.com/en/Crambin).

In [ ]:
crambin_alignment = ['TTCCPSIVARSNFNVCRLPGTPEALCATYTGCIIIPGATCPGDYAN',
                     'KSCCPTTAARNQYNICRLPGTPRPVCAALSGCKIISGTGCPPGYRH',
                     'KSCCPTTTARNIYNTCRFGGGSRPVCAKLSGCKIISGTKCDSGWNH',
                     'NICCPSIQARTFYNACLFAVGSPSSCIRNSSCLDISESTCPRGYTN',
                     'KSCCRNTLARNCYNACRFTGGSQPTCGILCDCIHVTTTTCPSSHPS',
                     'KSCCRSTLGRNCYNLCRVRGAQ-KLCANACRCKLTSGLKCPSSFPK',
                     'KSCCRSTLGRNCYNLCRVRGAQ-KLCAGVCRCKLTSSGKCPTGFPK',
                     'KICCPSNQARNGYSVCRIRF-SKGRCMQVSGCQ--NSDTCPRGWVN']

Here is a function to go through the columns of the alignment and make a sequence profile. The profile has. for each column in the MSA, a dictionary of residues types and the values for the fraction of the sequences containing them.

In [ ]:
def profile(alignment):
    """""
    calculate profile from an alignement
    """""
    columns = len(alignment[0])
    nSeq = float(len(alignment))
    prof = []
    for index in range(columns):
        counts = {}
        for seq in alignment:
            letter = seq[index]
            if letter == '-':
                continue

            counts[letter] = counts.get(letter, 0) + 1

        for letter in counts:
            counts[letter] /= nSeq
        prof.append(counts)
    return prof

Here is an example of the first item in the crambin alignment profile.

In [ ]:
exampleProfile = profile(crambin_alignment)
print(exampleProfile[1])

### Consensus sequence
The consensus sequence is the majority vote for residue type at each position. For large alignments with many sequences the majority might just be two outvoting 19 other different residues - so not really significant. So it makes sense to have a threshold below which values are not taken to be significant. Here it is set at 50%. The consensus sequence could also be calculated from the alignment profile. But here, more simply, the alignment is processed.

In [ ]:
def consensus(alignment, threshold=0.50):
    """""
    calculates a consensus sequence from
    a multiple sequence alignment
    """""
    numColumns = len(alignment[0])
    nSeq = len(alignment)
    consensus = ''

    for i in range(numColumns):
        counts = {}

        for seq in alignment:
            letter = seq[i]
            if letter == '-':
                continue

            counts[letter] = counts.get(letter, 0) + 1

        fractions = []
        for letter in counts:
            frac = counts[letter]/nSeq
            fractions.append([frac, letter])

        fractions.sort()
        bestFraction, bestLetter = fractions[-1]

        if bestFraction <  threshold:
            consensus += 'X'

        else:
            consensus += bestLetter

    return consensus


Use the consensus function to display the consensus sequence under the `crambin_alignment` defined  earlier.

In [ ]:
##your code here

The consensus sequence is a simple count that can isolate the key features of a sequence alignment - it might help identify a motif in a family of sequences for example.

However a better assessement of conservation and substitution would be to use Substitution table values. These can be applied to the set of residues in each column available in a profile.  

The following function gets a Conservation measure for each column in a multiple sequence alignment by making a profile then sorting the residues at each position in the profile.

In [ ]:
def getConservation(align, substTable):
    """""
    calculates the degree of conservation
    at each position in a multiple sequence alignment
    """""
    conservation = []
    profileAlignment = profile(align)

    for freqDict in profileAlignment:

        items = list(freqDict.items())

        items.sort( key=lambda x: x[1] )

        score = 0.0

        for resA, compA in items:
            for resB, compB in items:
                score += compA * compB * substTable[resA][resB]

        bestLetter = items[-1][0]
        maxScore = substTable[bestLetter][bestLetter]

        score /= maxScore
        conservation.append(score)

    return conservation

One common way to show conservation is to add a special final line below the alignment with symbols indicating increasing degrees of conservation. The symbols '*', ':', and '.' are commonly used to indicate perfect, high and moderate conservation respectively. No significant conservation will just be a space.

In [ ]:
def makeSimilarityLine(align, substTable, thresholds):
    """""
    makses a line of characters to stand for
    the degree of conservation in a multiple
    sequence alignment
    """""
    simString = ''
    conservation = getConservation(align, substTable)
    t1, t2, t3 = thresholds

    for score in conservation:

        if score >= t1:
            symbol = '*'
        elif score > t2:
            symbol = ':'
        elif score > t3:
            symbol = '.'
        else:
            symbol = ' '
        simString += symbol
    return simString


In [ ]:
conservationLine = makeSimilarityLine(crambin_alignment, BLOSUM62, (0.9,0.5,0.25))
for seq in crambin_alignment:
    print(seq)
print(conservationLine)

Some common residues are not considered significantly conserved even when they are the majority in the consensus.

##  Sum of pairs scoring for a multiple alignment
Sum of pairs scoring is a scoring function for a multiple sequence alignment. This is a common measure for the quality of a multiple sequence alignment. It is the overall conservation that the alignment has managed to maximize. This is worked out for all the pairs of residues for every pairing of the sequences.

So for an alignment:

    ADEH
    B-FI
    C-G-

for the first column it is the sum `substTable[A][B] + substTable[A][C]  + substTable[B][C]`.

For the second column it is the sum  `gapPenalty[D,-] + gapPenalty[D,-] + 0` (as two gaps aligned are set to 0).

For the third column it is `substTable[H][I] + gappenalty[H,-]  + gappenalty[I,-]`

Obviously the gap penalties will be negative values and the same regardless of the character being compared with it.

The total score is the sum of these totals of pairwise scores for each column in the alignment. Notice that it depends on the number of sequences so it is not for comparison of different alignments. It is for comparing different possible answers from the alignment process for a single set of sequences.

This is a useful target for the alignment algorithm to try to maximize.

Write a function to calculate the total score for the Crambin Alignment. **Hint** you could reuse the calcPairwiseSimilarityGapped function you wrote above to look at pairs of sequences.

In [ ]:
# your code for score_multiple_alingment

In [ ]:
# run this cell to check score_multiple_alignment function works as it should
assert score_multiple_alignment(['AW',
                                 'CP']) == BLOSUM62['A']['C'] + BLOSUM62['W']['P']
assert score_multiple_alignment(['A',
                                 'R']) == BLOSUM62['A']['R']
assert score_multiple_alignment(['A',
                                 'C',
                                 'E']) == BLOSUM62['A']['C'] + BLOSUM62['A']['E'] + BLOSUM62['C']['E']
assert score_multiple_alignment(['AW',
                                 'CP']) == BLOSUM62['A']['C'] + BLOSUM62['W']['P']
assert score_multiple_alignment(['AW',
                                 'CP',
                                 'EY']) == (BLOSUM62['A']['C'] + BLOSUM62['A']['E'] + BLOSUM62['C']['E'] +
                                            BLOSUM62['W']['P'] + BLOSUM62['W']['Y'] + BLOSUM62['P']['Y'])
assert score_multiple_alignment(['A',
                                 'C',
                                 '-']) == BLOSUM62['A']['C'] + BLOSUM62['A']['E'] - 1
assert score_multiple_alignment(['AWA',
                                 'CP-',
                                 'EY-']) == (BLOSUM62['A']['C'] + BLOSUM62['A']['E'] + BLOSUM62['C']['E'] +
                                                       BLOSUM62['W']['P'] + BLOSUM62['W']['Y'] + BLOSUM62['P']['Y']
                                                      -1 -1 + 0) # gap-gap is zero
assert score_multiple_alignment(crambin_alignment) == 2964

In [ ]:
# run this cell to find the total score of the crambin_alignment
print(score_multiple_alignment(crambin_alignment))